Ten skrypt oczyszcza i transformuje dane

In [0]:
%sql
MERGE INTO staging.car AS car
USING (
  SELECT
    Inventory.inventory_id,
    car.car_id,
    Car.producer,
    Car.model,
    Inventory.production_year,
    Inventory.fuel_type,
    Inventory.license_plates,
    Car.rental_rate AS current_rental_rate,
    Inventory.purchase_price,
    Inventory.sell_price,
    CASE WHEN Car.last_update > Inventory.last_update THEN Car.last_update ELSE Inventory.last_update END AS new_valid_to
  FROM landing.car AS Car
  INNER JOIN landing.inventory AS Inventory ON Car.car_id = Inventory.car_id
) AS src
ON car.car_inventory_id = src.inventory_id 
WHEN MATCHED AND (md5(CONCAT(COALESCE(car.car_producer,''),COALESCE(car.car_model,''),COALESCE(car.car_production_year,''),COALESCE(car.car_fuel_type,''),COALESCE(car.car_license_plates,''),COALESCE(car.car_current_rental_rate,0),COALESCE(car.car_purchase_price,0),COALESCE(car.car_sell_price,0))) != md5(CONCAT(COALESCE(src.producer,''),COALESCE(src.model,''),COALESCE(src.production_year,''),COALESCE(src.fuel_type,''),COALESCE(src.license_plates,''),COALESCE(src.current_rental_rate,0),COALESCE(src.purchase_price,0),COALESCE(src.sell_price,0))))
  THEN UPDATE SET  
    car.car_id = src.car_id,
    car.car_producer = src.producer,
    car.car_model = src.model,
    car.car_production_year = src.production_year,
    car.car_fuel_type = src.fuel_type,
    car.car_license_plates = src.license_plates,
    car.car_current_rental_rate = src.current_rental_rate,
    car.car_purchase_price = src.purchase_price,
    car.car_sell_price = src.sell_price,
    car.last_update=src.new_valid_to,
    car.stag_update_date = CURRENT_TIMESTAMP()
WHEN NOT MATCHED
  THEN INSERT(car_inventory_id, car_id, car_producer, car_model, car_production_year, car_fuel_type, car_license_plates, car_current_rental_rate, car_purchase_price, car_sell_price, last_update,stag_insert_date)    
  VALUES(src.inventory_id, src.car_id, src.producer, src.model, src.production_year, src.fuel_type, src.license_plates, src.current_rental_rate, src.purchase_price, src.sell_price, src.new_valid_to, CURRENT_TIMESTAMP());

In [0]:
%sql
UPDATE staging.car
SET car_fuel_type = 
  CASE LEFT(LOWER(car_fuel_type), 1)
    WHEN 'd' THEN 'Diesel'
    WHEN 'p' THEN 'Petrol'
    ELSE NULL END
WHERE car_fuel_type not in ('Diesel', 'Petrol');

In [0]:
%sql
UPDATE staging.car
SET car_purchase_price = 111000
WHERE car_purchase_price = 111;

In [0]:
%sql
UPDATE staging.car
SET car_production_year =  YEAR(current_timestamp())
WHERE car_production_year IS NULL OR car_production_year> YEAR(current_timestamp());

In [0]:
%sql
  CREATE OR REPLACE TABLE staging.equipment_groups AS
  SELECT ROW_NUMBER() OVER (ORDER BY eq_ids, eq_group_names) AS eq_group_id,
  eq_ids, 
  eq_group_names
  FROM (
  SELECT distinct eq_ids,eq_group_names
  FROM
    (
    SELECT
    inventory_id,
    string_agg(CAST(equipment_id AS STRING), ', ') WITHIN GROUP (ORDER BY equipment_id ASC) AS eq_ids,
    string_agg(CAST(eq_name AS STRING), ', ') WITHIN GROUP (ORDER BY eq_name ASC) AS eq_group_names
    FROM (
      SELECT ie.inventory_id, ie.equipment_id, CONCAT(name, ' ', type, COALESCE(version, '')) AS eq_name
      FROM landing.inventory_equipment ie
      INNER JOIN landing.equipment e ON ie.equipment_id = e.equipment_id
      ) t
    GROUP BY inventory_id
    ) a
  ) d


In [0]:
%sql
CREATE OR REPLACE TABLE staging.bridge_equipment__equipment_groups AS
SELECT
  eg.eq_group_id,
  eg.eq_group_names,
  CAST(TRIM(equipment_id) AS INT) AS equipment_id
FROM staging.equipment_groups eg
LATERAL VIEW EXPLODE(SPLIT(eg.eq_ids, ', ')) AS equipment_id;

In [0]:
%sql
CREATE OR REPLACE TABLE staging.inventory_equipment_groups AS
SELECT i.inventory_id, eq_group.eq_group_id
FROM (
    SELECT 
        inventory_id,
        string_agg(CAST(equipment_id AS STRING), ', ') WITHIN GROUP (ORDER BY equipment_id ASC) AS eq_ids,
        string_agg(CAST(eq_name AS STRING), ', ') WITHIN GROUP (ORDER BY eq_name ASC) AS eq_group_names
    FROM (
      SELECT ie.inventory_id, ie.equipment_id, CONCAT(name, ' ', type, COALESCE(version, '')) AS eq_name
      FROM landing.inventory_equipment ie
      INNER JOIN landing.equipment e ON ie.equipment_id = e.equipment_id
      ) t
    GROUP BY inventory_id
) i
LEFT JOIN staging.equipment_groups eq_group ON i.eq_ids = eq_group.eq_ids